# ==============================
# Feature Engineering for ML
# ==============================

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('../data/raw/cleaned_staten_island_otp.csv')
df['Month'] = pd.to_datetime(df['Month'])

df.head()

,Month,Day Time,Delayed Trains,On-Time Trips,On-Time Performance,Delayed Trains (With Boat),On-Time Trips (With Boat),On-Time Performance (With Boat),Scheduled Trips,Incomplete Trips,Trip Complete Percentage,Year,Month_Number,Month_Name
0,2006-01-01,Weekday,83,2621,0.969,124,2580,0.954,2704,NaN,NaN,2006,1,January
1,2006-01-01,AM Rush,21,672,0.970,24,669,0.965,693,NaN,NaN,2006,1,January
2,2006-01-01,PM Rush,10,830,0.988,34,806,0.960,840,NaN,NaN,2006,1,January
3,2006-01-01,Weekend,96,722,0.883,103,715,0.874,818,NaN,NaN,2006,1,January
4,2006-01-01,7-Day,179,3343,0.949,227,3295,0.936,3522,10.0,99.7,2006,1,January


In [3]:
df = df.sort_values(['Day Time', 'Month']).reset_index(drop=True)

Creating time features:

In [4]:
df['Year'] = df['Month'].dt.year
df['Month_Number'] = df['Month'].dt.month
df['Quarter'] = df['Month'].dt.quarter

Creating Season features:

In [5]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['Season'] = df['Month_Number'].apply(get_season)

Creating lag features.

In [7]:
df['OTP_Lag_1'] = df.groupby('Day Time')['On-Time Performance'].shift(1)
df['OTP_Lag_2'] = df.groupby('Day Time')['On-Time Performance'].shift(2)
df['OTP_Lag_3'] = df.groupby('Day Time')['On-Time Performance'].shift(3)

df['Delayed_Trains_Lag_1'] = df.groupby('Day Time')['Delayed Trains'].shift(1)
df['Scheduled_Trips_Lag_1'] = df.groupby('Day Time')['Scheduled Trips'].shift(1)

Creating Rolling average features

In [8]:
df['OTP_Rolling_3'] = (
    df.groupby('Day Time')['On-Time Performance']
    .shift(1)
    .rolling(window=3)
    .mean()
)

df['OTP_Rolling_6'] = (
    df.groupby('Day Time')['On-Time Performance']
    .shift(1)
    .rolling(window=6)
    .mean()
)

df['Delayed_Trains_Rolling_3'] = (
    df.groupby('Day Time')['Delayed Trains']
    .shift(1)
    .rolling(window=3)
    .mean()
)

Delay rates

In [9]:
df['Delay_Rate'] = df['Delayed Trains'] / df['Scheduled Trips']

Next month OTP

In [10]:
df['Next_Month_OTP'] = df.groupby('Day Time')['On-Time Performance'].shift(-1)

In [11]:
ml_df = df.dropna(subset=[
    'OTP_Lag_1',
    'OTP_Lag_2',
    'OTP_Lag_3',
    'OTP_Rolling_3',
    'OTP_Rolling_6',
    'Delayed_Trains_Lag_1',
    'Delayed_Trains_Rolling_3',
    'Next_Month_OTP'
]).copy()

In [12]:
ml_df.shape

(1170, 26)

In [13]:
ml_df.head()

,Month,Day Time,Delayed Trains,On-Time Trips,On-Time Performance,Delayed Trains (With Boat),On-Time Trips (With Boat),On-Time Performance (With Boat),Scheduled Trips,Incomplete Trips,...,OTP_Lag_1,OTP_Lag_2,OTP_Lag_3,Delayed_Trains_Lag_1,Scheduled_Trips_Lag_1,OTP_Rolling_3,OTP_Rolling_6,Delayed_Trains_Rolling_3,Delay_Rate,Next_Month_OTP
6,2006-07-01,7-Day,104,3362,0.970,192,3274,0.945,3466,26.0,...,0.983,0.968,0.965,61.0,3490.0,0.972000,0.974833,97.666667,0.030006,0.971
7,2006-08-01,7-Day,105,3506,0.971,160,3451,0.956,3611,0.0,...,0.970,0.983,0.968,104.0,3466.0,0.973667,0.978333,92.666667,0.029078,0.953
8,2006-09-01,7-Day,159,3228,0.953,220,3167,0.935,3387,0.0,...,0.971,0.970,0.983,105.0,3611.0,0.974667,0.976167,90.000000,0.046944,0.955
9,2006-10-01,7-Day,159,3401,0.955,220,3340,0.938,3560,38.0,...,0.953,0.971,0.970,159.0,3387.0,0.964667,0.968333,122.666667,0.044663,0.930
10,2006-11-01,7-Day,239,3197,0.930,303,3133,0.912,3436,20.0,...,0.955,0.953,0.971,159.0,3560.0,0.959667,0.966667,141.000000,0.069558,0.980


In [14]:
ml_df.to_csv('../outputs/predictions/staten_island_otp_features.csv', index=False)